# 04 – Temporal 5-fold Cross-Validation

Evaluates temporal generalisation by withholding 4 non-consecutive years per
fold.  The model is trained on the remaining years and evaluated on the
held-out years, testing generalisation to unseen time periods.

The year assignments below were generated by random selection from the
20-year study period (2001–2020), stratified to spread held-out years
across the full time range rather than grouping them consecutively.

| Fold | Held-out years |
|------|---------------|
| 1 | 2001, 2006, 2013, 2014 |
| 2 | 2003, 2008, 2012, 2018 |
| 3 | 2004, 2010, 2011, 2016 |
| 4 | 2002, 2015, 2019, 2020 |
| 5 | 2005, 2007, 2009, 2017 |

**Prerequisites:** `01_data_preparation.ipynb`

**Outputs** (in `<output_dir>/`):
- `fold<k>_best.pt`, `fold<k>_last.pt` – checkpoints per fold
- `5fold_temporal_results.json` – losses and R² per fold and epoch

In [ ]:
import os, random, json, warnings, pickle
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import SequentialLR, LinearLR, StepLR
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

from sparrow.model import SPARROW, ParamGenerator
from sparrow.utils import setup_seed, gpu_align, r2_torch, r2_logspace_torch

# ── User settings ─────────────────────────────────────────────────────────
prepared_dir = './outputs/prepared/'
output_dir   = './outputs/temporal_cv/'
os.makedirs(output_dir, exist_ok=True)

NUM_EPOCHS = 150
SEED       = 42
setup_seed(SEED)

# Year assignments generated by random selection; each fold holds out
# 4 non-consecutive years spread across the 20-year study period.
FOLD_VAL_YEARS = {
    1: [2001, 2006, 2013, 2014],
    2: [2003, 2008, 2012, 2018],
    3: [2004, 2010, 2011, 2016],
    4: [2002, 2015, 2019, 2020],
    5: [2005, 2007, 2009, 2017],
}


In [ ]:
with open(os.path.join(prepared_dir, 'prepared_data.pkl'), 'rb') as f:
    prep = pickle.load(f)

input_data_huc12   = prep['input_data_huc12']
dlvdsgn_array      = prep['dlvdsgn_array']
source_columns     = prep['source_columns'];  delivery_columns  = prep['delivery_columns']
stream_columns     = prep['stream_columns'];  reservoir_columns = prep['reservoir_columns']
input_columns      = prep['input_columns']
input_columns_strm = prep['input_columns_strm']
input_columns_res  = prep['input_columns_res']
scaler = prep['scaler']; scaler_strm = prep['scaler_strm']; scaler_res = prep['scaler_res']
years  = prep['years']

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dlvdsgn = torch.tensor(dlvdsgn_array, dtype=torch.float32).to(device)
print(f"Device: {device}")

ns = len(source_columns); nd = len(delivery_columns)


In [ ]:
# Build yearly DataLoaders
yearly_loaders = []
for year in years:
    yd = input_data_huc12[input_data_huc12['Year'] == year].copy()
    X_sc      = torch.tensor(scaler.transform(yd[input_columns].values),
                             dtype=torch.float32, device=device)
    X_sc_strm = torch.tensor(scaler_strm.transform(yd[input_columns_strm].values),
                             dtype=torch.float32, device=device)
    X_sc_res  = torch.tensor(scaler_res.transform(yd[input_columns_res].values),
                             dtype=torch.float32, device=device)
    obs = torch.tensor(yd['depvar'].values, dtype=torch.float32, device=device)
    obs[obs == 0] = float('nan')
    ds = TensorDataset(
        torch.tensor(yd[input_columns].values,       dtype=torch.float32, device=device),
        X_sc, X_sc_strm, X_sc_res,
        torch.tensor(yd[source_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[delivery_columns].values,  dtype=torch.float32, device=device),
        torch.tensor(yd[stream_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[reservoir_columns].values, dtype=torch.float32, device=device),
        torch.tensor(yd['rchtype'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['hydseq'].values,     dtype=torch.float32, device=device),
        torch.tensor(yd['headflag'].values,   dtype=torch.int64,   device=device),
        torch.tensor(yd['from_node'].values,  dtype=torch.int64,   device=device),
        torch.tensor(yd['to_node'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['frac'].values,       dtype=torch.float32, device=device),
        torch.tensor(yd['new_waterid'].values,dtype=torch.int64,   device=device),
        torch.tensor(yd['waterid'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['iftran'].values,     dtype=torch.int64,   device=device),
        obs,
    )
    yearly_loaders.append(DataLoader(ds, batch_size=len(yd), shuffle=False))


## 5-fold temporal cross-validation

In [ ]:
save_dir     = Path(output_dir)
criterion    = nn.MSELoss()
fold_results = []

for fold, val_years in FOLD_VAL_YEARS.items():
    print(f"\n=== Temporal Fold {fold}/5  (val years: {val_years}) ===")

    train_loaders_ = [ldr for yr, ldr in zip(years, yearly_loaders) if yr not in val_years]
    val_loaders_   = [ldr for yr, ldr in zip(years, yearly_loaders) if yr in val_years]

    param_model = ParamGenerator(
        input_size=len(input_columns), hidden_size=32,
        num_source=ns, num_delivery=nd,
        num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
    ).to(device)
    param_model_strm = ParamGenerator(
        input_size=len(input_columns_strm), hidden_size=8,
        num_source=ns, num_delivery=nd,
        num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
    ).to(device)
    param_model_res = ParamGenerator(
        input_size=len(input_columns_res), hidden_size=8,
        num_source=ns, num_delivery=nd,
        num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
    ).to(device)
    sparrow_model = SPARROW().to(device)

    optimizer = optim.Adam(
        list(param_model.parameters())
        + list(param_model_strm.parameters())
        + list(param_model_res.parameters()),
        lr=1e-3, weight_decay=1e-4)
    scheduler = SequentialLR(
        optimizer,
        schedulers=[LinearLR(optimizer, start_factor=0.1, total_iters=10),
                    StepLR(optimizer, step_size=50, gamma=0.5)],
        milestones=[10])

    train_losses, val_losses = [], []
    fold_train_r2_log, fold_val_r2_log, fold_val_r2_lin = [], [], []
    best_val = float('inf')

    def _run_epoch(loaders, train=True):
        total_loss = 0.0
        preds_list, obs_list = [], []
        shuffled = loaders.copy()
        if train:
            random.shuffle(shuffled)
        for loader in shuffled:
            for batch in loader:
                (X_raw, X_sc, X_sc_strm, X_sc_res,
                 S, Z_D, Z_S, Z_R,
                 rtype, hseq, hflag, fnode, tnode, frac,
                 cid, ocid, iftran, obs) = [b.to(device) for b in batch]

                coeffs      = param_model(X_sc)
                coeffs_strm = param_model_strm(X_sc_strm)
                coeffs_res  = param_model_res(X_sc_res)
                alpha   = coeffs[:, :ns]
                theta_D = coeffs[:, ns:ns + nd]
                theta_S = coeffs_strm[:, -2:-1]
                theta_R = coeffs_res[:, -1:]

                _, _, _, total_load, ocid_out = sparrow_model(
                    S, Z_D, Z_S, Z_R, rtype, hseq, hflag, fnode, tnode,
                    alpha, theta_D, theta_S, theta_R,
                    frac, cid, ocid, dlvdsgn, iftran)
                total_load = total_load.to(device)
                pred, obs_a, mask = gpu_align(total_load, obs, ocid_out, ocid)

                if mask.sum() > 0:
                    loss = criterion(
                        torch.log(torch.clamp(pred[mask],  min=1e-6)),
                        torch.log(torch.clamp(obs_a[mask], min=1e-6)))
                    if train:
                        optimizer.zero_grad(); loss.backward(); optimizer.step()
                    total_loss += loss.item()
                    preds_list.append(pred[mask].detach().cpu())
                    obs_list.append(obs_a[mask].detach().cpu())
        avg = total_loss / len(loaders) if loaders else float('nan')
        return avg, preds_list, obs_list

    for epoch in range(NUM_EPOCHS):
        param_model.train(); param_model_strm.train(); param_model_res.train()
        avg_train, tp, to_ = _run_epoch(train_loaders_, train=True)

        param_model.eval(); param_model_strm.eval(); param_model_res.eval()
        with torch.no_grad():
            avg_val, vp, vo = _run_epoch(val_loaders_, train=False)

        train_losses.append(avg_train); val_losses.append(avg_val)

        r2_tr  = float(r2_logspace_torch(torch.cat(to_), torch.cat(tp))) if tp else float('nan')
        r2_vl  = float(r2_logspace_torch(torch.cat(vo),  torch.cat(vp))) if vp else float('nan')
        r2_vlin= float(r2_torch(torch.cat(vo),           torch.cat(vp))) if vp else float('nan')
        fold_train_r2_log.append(r2_tr)
        fold_val_r2_log.append(r2_vl)
        fold_val_r2_lin.append(r2_vlin)

        ckpt = {'epoch': epoch+1,
                'model_state_dict':      param_model.state_dict(),
                'model_strm_state_dict': param_model_strm.state_dict(),
                'model_res_state_dict':  param_model_res.state_dict(),
                'optimizer_state_dict':  optimizer.state_dict(),
                'scheduler_state_dict':  scheduler.state_dict(),
                'val_loss': avg_val, 'r2_val_log': r2_vl, 'r2_val_linear': r2_vlin}
        torch.save(ckpt, save_dir / f'fold{fold}_last.pt')
        if avg_val < best_val:
            best_val = avg_val
            torch.save(ckpt, save_dir / f'fold{fold}_best.pt')

        scheduler.step()
        print(f"Epoch {epoch+1:3d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
              f"R²(log): {r2_vl:.4f}")

    fold_results.append({
        'fold': fold, 'val_years': val_years,
        'train_losses': train_losses, 'test_losses': val_losses,
        'train_r2_log': fold_train_r2_log,
        'val_r2_log':   fold_val_r2_log,
        'val_r2_linear': fold_val_r2_lin,
        'final_test_loss': val_losses[-1],
    })

with open(os.path.join(output_dir, '5fold_temporal_results.json'), 'w') as f:
    json.dump({'folds': fold_results}, f, indent=2)
print("\nAll folds complete. Results saved.")


## Summary

In [ ]:
final_r2  = [r['val_r2_log'][-1]   for r in fold_results]
final_r2l = [r['val_r2_linear'][-1] for r in fold_results]
final_mse = [r['test_losses'][-1]   for r in fold_results]
print(f"Val R² (log)    per fold: {[f'{v:.3f}' for v in final_r2]}")
print(f"  Mean ± SD: {np.mean(final_r2):.3f} ± {np.std(final_r2):.3f}")
print(f"Val R² (linear) per fold: {[f'{v:.3f}' for v in final_r2l]}")
print(f"  Mean ± SD: {np.mean(final_r2l):.3f} ± {np.std(final_r2l):.3f}")
print(f"Val MSE         per fold: {[f'{v:.4f}' for v in final_mse]}")
print(f"  Mean ± SD: {np.mean(final_mse):.4f} ± {np.std(final_mse):.4f}")
